In [1]:
from google.colab import drive
drive.mount('/content/drive')

In [4]:
from pathlib import Path
from PIL import Image

root_path = Path("/content/drive/MyDrive/coco_dataset")

exten = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


In [5]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import json

# Dataset


In [6]:
class CocoDetectionDataset(Dataset):
    def __init__(self, root: Path, split: str, image_size: int = 512, cat_id_to_contig=None):
        self.root = Path(root)
        self.split = split
        self.img_dir = self.root / split
        self.ann_path = self.img_dir / "_annotations.coco.json"
        if not self.ann_path.exists():
            raise FileNotFoundError(f"Missing: {self.ann_path}")

        with open(self.ann_path, "r") as f:
            coco = json.load(f)

        categories = coco.get("categories", [])
        if cat_id_to_contig is None:
            cat_ids = sorted([c["id"] for c in categories])
            self.cat_id_to_contig = {cid: i for i, cid in enumerate(cat_ids)}
        else:
            self.cat_id_to_contig = dict(cat_id_to_contig)

        self.contig_to_name = {}
        for c in categories:
            cid = c["id"]
            if cid in self.cat_id_to_contig:
                self.contig_to_name[self.cat_id_to_contig[cid]] = c.get("name", str(cid))

        self.images = coco.get("images", [])
        if not self.images:
            raise ValueError(f"No images in {self.ann_path}")

        self.image_id_to_info = {im["id"]: im for im in self.images}
        self.image_ids = sorted(self.image_id_to_info.keys())

        self.ann_by_image = {iid: [] for iid in self.image_ids}
        for ann in coco.get("annotations", []):
            if ann.get("iscrowd", 0) == 1:
                continue
            iid = ann["image_id"]
            if iid in self.ann_by_image:
                self.ann_by_image[iid].append(ann)

        self.image_size = int(image_size)
        self.to_tensor = transforms.ToTensor()

    @property
    def num_classes(self):
        return len(set(self.cat_id_to_contig.values()))

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        info = self.image_id_to_info[image_id]
        file_name = info["file_name"]
        img_path = self.img_dir / file_name
        if not img_path.exists():
            raise FileNotFoundError(f"Image missing: {img_path}")

        img = Image.open(img_path).convert("RGB")
        W0, H0 = img.size

        anns = self.ann_by_image.get(image_id, [])
        boxes = []
        labels = []

        for a in anns:
            cat_id = a["category_id"]
            if cat_id not in self.cat_id_to_contig:
                continue

            x, y, w, h = a["bbox"]
            if w <= 0 or h <= 0:
                continue

            x1 = x
            y1 = y
            x2 = x + w
            y2 = y + h

            x1 = max(0.0, min(x1, W0 - 1))
            y1 = max(0.0, min(y1, H0 - 1))
            x2 = max(0.0, min(x2, W0 - 1))
            y2 = max(0.0, min(y2, H0 - 1))
            if x2 <= x1 or y2 <= y1:
                continue

            boxes.append([x1, y1, x2, y2])
            labels.append(self.cat_id_to_contig[cat_id])

        img = img.resize((self.image_size, self.image_size), resample=Image.BILINEAR)
        sx = self.image_size / W0
        sy = self.image_size / H0

        if boxes:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            boxes[:, [0, 2]] *= sx
            boxes[:, [1, 3]] *= sy
            labels = torch.tensor(labels, dtype=torch.long)
        else:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.long)

        image_tensor = self.to_tensor(img)
        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([image_id], dtype=torch.long),
        }
        return image_tensor, target


def detection_collate_fn(batch):
    images, targets = zip(*batch)
    return torch.stack(images, dim=0), list(targets)



In [7]:
image_size = 512

train_tmp = CocoDetectionDataset(root_path, "train", image_size=image_size, cat_id_to_contig=None)
categories_id = train_tmp.cat_id_to_contig

train_ds = CocoDetectionDataset(root_path, "train", image_size=image_size, cat_id_to_contig=categories_id)
val_ds   = CocoDetectionDataset(root_path, "valid",   image_size=image_size, cat_id_to_contig=categories_id)
test_ds  = CocoDetectionDataset(root_path, "test",  image_size=image_size, cat_id_to_contig=categories_id)

NUM_CLASSES = train_ds.num_classes
id_to_name = train_ds.contig_to_name

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,  num_workers=2, pin_memory=True,
                          collate_fn=detection_collate_fn)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=2, pin_memory=True,
                          collate_fn=detection_collate_fn)
test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False, num_workers=2, pin_memory=True,
                          collate_fn=detection_collate_fn)

print("NUM_CLASSES =", NUM_CLASSES)
print("Example id_to_name:", dict(list(id_to_name.items())[:5]))

In [8]:
id_to_name

# Build the Model

## CNN Backbone

In [9]:
import torch.nn as nn


class ConvBNReLU(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False)
        self.bn   = nn.BatchNorm2d(out_ch)
        self.act  = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))


class BasicResidualBlock(nn.Module):
    
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = ConvBNReLU(in_ch, out_ch, k=3, s=stride, p=1)
        self.conv2 = nn.Conv2d(out_ch, out_ch, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)

        self.proj = None
        if stride != 1 or in_ch != out_ch:
            self.proj = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=1, stride=stride, padding=0, bias=False),
                nn.BatchNorm2d(out_ch),
            )

        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        identity = x
        out = self.conv1(x)
        out = self.bn2(self.conv2(out))
        if self.proj is not None:
            identity = self.proj(identity)
        out = self.act(out + identity)
        return out


In [10]:
class TinyBackbone(nn.Module):

    def __init__(self, base_ch=32, in_ch=3):
        super().__init__()
        base_ch = int(base_ch)
        in_ch = int(in_ch)

        # stage channels
        ch2 = base_ch * 2
        ch3 = base_ch * 4
        ch4 = base_ch * 8
        ch5 = base_ch * 16

        self.out_channels = (ch3, ch4, ch5)
        self.out_strides  = (8, 16, 32)

        # Stem: /2
        self.stem = nn.Sequential(
            ConvBNReLU(in_ch, base_ch, k=3, s=2, p=1),
            ConvBNReLU(base_ch, base_ch, k=3, s=1, p=1),
        )

        # Stage2: /4
        self.stage2 = nn.Sequential(
            BasicResidualBlock(base_ch, ch2, stride=2),
            BasicResidualBlock(ch2, ch2, stride=1),
        )

        # Stage3: /8  -> C3
        self.stage3 = nn.Sequential(
            BasicResidualBlock(ch2, ch3, stride=2),
            BasicResidualBlock(ch3, ch3, stride=1),
        )

        # Stage4: /16 -> C4
        self.stage4 = nn.Sequential(
            BasicResidualBlock(ch3, ch4, stride=2),
            BasicResidualBlock(ch4, ch4, stride=1),
        )

        # Stage5: /32 -> C5
        self.stage5 = nn.Sequential(
            BasicResidualBlock(ch4, ch5, stride=2),
            BasicResidualBlock(ch5, ch5, stride=1),
        )

    def forward(self, x):
        x = self.stem(x)        # /2
        x = self.stage2(x)      # /4
        c3 = self.stage3(x)     # /8
        c4 = self.stage4(c3)    # /16
        c5 = self.stage5(c4)    # /32
        return c3, c4, c5

## FPN

In [11]:
import torch.nn.functional as F

class FPN(nn.Module):

    def __init__(self, in_channels, fpn_ch=128):

        super().__init__()
        c3_ch, c4_ch, c5_ch = map(int, in_channels)
        fpn_ch = int(fpn_ch)

        # 1x1 lateral convolutions 
        self.lat_c3 = nn.Conv2d(c3_ch, fpn_ch, kernel_size=1)
        self.lat_c4 = nn.Conv2d(c4_ch, fpn_ch, kernel_size=1)
        self.lat_c5 = nn.Conv2d(c5_ch, fpn_ch, kernel_size=1)

        # 3x3 convolutions 
        self.out_p3 = nn.Conv2d(fpn_ch, fpn_ch, kernel_size=3, padding=1)
        self.out_p4 = nn.Conv2d(fpn_ch, fpn_ch, kernel_size=3, padding=1)
        self.out_p5 = nn.Conv2d(fpn_ch, fpn_ch, kernel_size=3, padding=1)

        self.out_channels = (fpn_ch, fpn_ch, fpn_ch)

    def forward(self, c3, c4, c5):
        p5 = self.lat_c5(c5)
        p4 = self.lat_c4(c4) + F.interpolate(p5, size=c4.shape[-2:], mode="nearest")
        p3 = self.lat_c3(c3) + F.interpolate(p4, size=c3.shape[-2:], mode="nearest")

        p5 = self.out_p5(p5)
        p4 = self.out_p4(p4)
        p3 = self.out_p3(p3)
        return p3, p4, p5


## Detection Head

In [12]:
import math


class DetectionHead(nn.Module):
    def __init__(self, in_ch: int, num_classes: int, num_layers: int = 4):
        super().__init__()

        self.in_ch = int(in_ch)
        self.num_classes = int(num_classes)
        self.num_layers = int(num_layers)

        def make_tower():
            layers = []
            for _ in range(self.num_layers):
                layers.append(nn.Conv2d(self.in_ch, self.in_ch, kernel_size=3, padding=1))
                layers.append(nn.ReLU(inplace=True))
            return nn.Sequential(*layers)

        self.cls_tower = make_tower()
        self.reg_tower = make_tower()

        self.cls_logits = nn.Conv2d(self.in_ch, self.num_classes, kernel_size=3, padding=1)
        self.bbox_pred  = nn.Conv2d(self.in_ch, 4, kernel_size=3, padding=1)
        self.centerness = nn.Conv2d(self.in_ch, 1, kernel_size=3, padding=1)

        self._init_weights()

    def _init_weights(self):
        prior_prob = 0.01
        bias_value = -math.log((1 - prior_prob) / prior_prob)
        nn.init.constant_(self.cls_logits.bias, bias_value)

        for layer in [self.bbox_pred, self.centerness]:
            nn.init.constant_(layer.bias, 0.0)

    def forward(self, features):
        cls_outputs = []
        box_outputs = []
        ctr_outputs = []

        for x in features:
            cls_feat = self.cls_tower(x)
            reg_feat = self.reg_tower(x)

            cls_outputs.append(self.cls_logits(cls_feat))
            box_outputs.append(F.relu(self.bbox_pred(reg_feat)))
            ctr_outputs.append(self.centerness(reg_feat))

        return cls_outputs, box_outputs, ctr_outputs


In [ ]:
backbone = TinyBackbone(base_ch=32)
fpn = FPN(backbone.out_channels, fpn_ch=128)
head = DetectionHead(in_ch=fpn.out_channels[0],num_classes=NUM_CLASSES)


## Location Grid Generation

In [13]:
_location_cache = {}

def compute_locations(feature: torch.Tensor, stride: int) -> torch.Tensor:
    _, _, H, W = feature.shape
    device = feature.device
    key = (H, W, stride, device)

    if key in _location_cache:
        return _location_cache[key]

    ys = (torch.arange(H, device=device, dtype=torch.float32) + 0.5) * stride
    xs = (torch.arange(W, device=device, dtype=torch.float32) + 0.5) * stride

    yy, xx = torch.meshgrid(ys, xs, indexing="ij")
    locations = torch.stack((xx, yy), dim=-1).reshape(-1, 2)

    _location_cache[key] = locations
    return locations


# Target Assignment

In [14]:
def assign_targets_single_level(
    locations: torch.Tensor,     
    boxes_xyxy: torch.Tensor,   
    labels: torch.Tensor,        
    num_classes: int,
    reg_range=None,              
    stride: int = None,         
    center_sampling_radius: float = 1.5, 
):
    device = locations.device
    L = locations.shape[0]

    cls_tgt  = torch.full((L,), -1, dtype=torch.long, device=device)  
    reg_tgt  = torch.zeros((L, 4), dtype=torch.float32, device=device)
    ctr_tgt  = torch.zeros((L,), dtype=torch.float32, device=device)
    pos_mask = torch.zeros((L,), dtype=torch.bool, device=device)

    if boxes_xyxy.numel() == 0:
        return cls_tgt, reg_tgt, ctr_tgt, pos_mask

    if (labels.min() < 0) or (labels.max() >= num_classes):
        raise ValueError(f"Found label outside [0..{num_classes-1}]")

    xs = locations[:, 0][:, None] 
    ys = locations[:, 1][:, None]  

    x1 = boxes_xyxy[:, 0][None, :] 
    y1 = boxes_xyxy[:, 1][None, :]
    x2 = boxes_xyxy[:, 2][None, :]
    y2 = boxes_xyxy[:, 3][None, :]

    l = xs - x1
    t = ys - y1
    r = x2 - xs
    b = y2 - ys

    inside = (l >= 0) & (t >= 0) & (r >= 0) & (b >= 0)

    if stride is not None and center_sampling_radius is not None:
        cx = (x1 + x2) / 2.0
        cy = (y1 + y2) / 2.0
        radius = center_sampling_radius * stride

        x1c = torch.max(x1, cx - radius)
        y1c = torch.max(y1, cy - radius)
        x2c = torch.min(x2, cx + radius)
        y2c = torch.min(y2, cy + radius)

        inside_center = (xs >= x1c) & (xs <= x2c) & (ys >= y1c) & (ys <= y2c)
        inside = inside & inside_center

    if reg_range is not None:
        reg_min, reg_max = reg_range
        max_lrbt = torch.max(torch.stack([l, t, r, b], dim=0), dim=0).values 
        in_range = (max_lrbt >= reg_min) & (max_lrbt <= reg_max)
        inside = inside & in_range

    areas = (x2 - x1) * (y2 - y1)         
    areas = areas.expand(L, -1).clone()   
    areas[~inside] = 1e18

    min_area, min_idx = areas.min(dim=1)  
    assigned = min_area < 1e17
    pos_mask = assigned


    cls_tgt[pos_mask] = labels[min_idx[pos_mask]]


    sel = min_idx[pos_mask] 
    reg_tgt[pos_mask, 0] = l[pos_mask, sel]
    reg_tgt[pos_mask, 1] = t[pos_mask, sel]
    reg_tgt[pos_mask, 2] = r[pos_mask, sel]
    reg_tgt[pos_mask, 3] = b[pos_mask, sel]

    l_pos = reg_tgt[pos_mask, 0]
    t_pos = reg_tgt[pos_mask, 1]
    r_pos = reg_tgt[pos_mask, 2]
    b_pos = reg_tgt[pos_mask, 3]

    eps = 1e-6
    ctr = torch.sqrt(
        (torch.min(l_pos, r_pos) / (torch.max(l_pos, r_pos) + eps)) *
        (torch.min(t_pos, b_pos) / (torch.max(t_pos, b_pos) + eps))
    )
    ctr_tgt[pos_mask] = ctr

    return cls_tgt, reg_tgt, ctr_tgt, pos_mask


## Batch level Target builder

In [15]:
def default_reg_ranges_from_strides(strides, factor=8.0):
    strides = [float(s) for s in strides]
    maxs = [factor * s for s in strides]
    ranges = []
    prev = 0.0
    for i, mx in enumerate(maxs):
        if i == len(maxs) - 1:
            ranges.append((prev, float("inf")))
        else:
            ranges.append((prev, mx))
            prev = mx
    return ranges


def build_targets_batch(
    features,                 
    targets,                  
    strides,                  
    num_classes: int,
    reg_ranges=None,          
    center_sampling_radius=1.5,
):
    device = features[0].device
    B = len(targets)

    if reg_ranges is None:
        reg_ranges = default_reg_ranges_from_strides(strides, factor=8.0)

    assert len(features) == len(strides) == len(reg_ranges), "features/strides/reg_ranges must match in length"

    cls_targets, reg_targets, ctr_targets, pos_masks = [], [], [], []

    for feat, stride, reg_range in zip(features, strides, reg_ranges):
        _, _, H, W = feat.shape
        L = H * W

        locations = compute_locations(feat, stride)

        cls_lvl = torch.full((B, L), -1, dtype=torch.long, device=device)
        reg_lvl = torch.zeros((B, L, 4), dtype=torch.float32, device=device)
        ctr_lvl = torch.zeros((B, L), dtype=torch.float32, device=device)
        pos_lvl = torch.zeros((B, L), dtype=torch.bool, device=device)

        for b in range(B):
            boxes  = targets[b]["boxes"].to(device)
            labels = targets[b]["labels"].to(device)

            cls_b, reg_b, ctr_b, pos_b = assign_targets_single_level(
                locations=locations,
                boxes_xyxy=boxes,
                labels=labels,
                num_classes=num_classes,
                reg_range=reg_range,
                stride=stride,
                center_sampling_radius=center_sampling_radius,
            )

            cls_lvl[b] = cls_b
            reg_lvl[b] = reg_b
            ctr_lvl[b] = ctr_b
            pos_lvl[b] = pos_b

        cls_targets.append(cls_lvl.view(B, H, W))
        reg_targets.append(reg_lvl.view(B, H, W, 4))
        ctr_targets.append(ctr_lvl.view(B, H, W))
        pos_masks.append(pos_lvl.view(B, H, W))

    return cls_targets, reg_targets, ctr_targets, pos_masks


#Loss Function

In [16]:
def ltrb_to_xyxy(locations, ltrb):
    x = locations[:, 0]
    y = locations[:, 1]
    l = ltrb[:, 0]
    t = ltrb[:, 1]
    r = ltrb[:, 2]
    b = ltrb[:, 3]
    return torch.stack([x - l, y - t, x + r, y + b], dim=1)


def giou_loss(boxes1, boxes2, reduction="mean", eps=1e-6):
    x1 = torch.max(boxes1[:, 0], boxes2[:, 0])
    y1 = torch.max(boxes1[:, 1], boxes2[:, 1])
    x2 = torch.min(boxes1[:, 2], boxes2[:, 2])
    y2 = torch.min(boxes1[:, 3], boxes2[:, 3])
    inter = (x2 - x1).clamp(min=0) * (y2 - y1).clamp(min=0)

    area1 = (boxes1[:, 2] - boxes1[:, 0]).clamp(min=0) * (boxes1[:, 3] - boxes1[:, 1]).clamp(min=0)
    area2 = (boxes2[:, 2] - boxes2[:, 0]).clamp(min=0) * (boxes2[:, 3] - boxes2[:, 1]).clamp(min=0)

    union = area1 + area2 - inter + eps
    iou = inter / union

    cx1 = torch.min(boxes1[:, 0], boxes2[:, 0])
    cy1 = torch.min(boxes1[:, 1], boxes2[:, 1])
    cx2 = torch.max(boxes1[:, 2], boxes2[:, 2])
    cy2 = torch.max(boxes1[:, 3], boxes2[:, 3])
    c_area = (cx2 - cx1).clamp(min=0) * (cy2 - cy1).clamp(min=0) + eps

    giou = iou - (c_area - union) / c_area
    loss = 1.0 - giou

    if reduction == "none":
        return loss
    elif reduction == "sum":
        return loss.sum()
    else:
        return loss.mean()


In [17]:
def sigmoid_focal_loss(logits, targets, alpha=0.25, gamma=2.0, reduction="sum"):
    prob = torch.sigmoid(logits)
    ce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")  # [N,C]
    p_t = prob * targets + (1 - prob) * (1 - targets)
    modulating = (1 - p_t).pow(gamma)
    alpha_t = alpha * targets + (1 - alpha) * (1 - targets)
    loss = alpha_t * modulating * ce

    if reduction == "sum":
        return loss.sum()
    elif reduction == "mean":
        return loss.mean()
    else:
        return loss

def compute_losses_per_level(
    cls_pred,   
    box_pred,   
    ctr_pred,   
    cls_tgt,    
    reg_tgt,    
    ctr_tgt,    
    pos_mask,   
    locations, 
):
    device = cls_pred.device
    B, C, H, W = cls_pred.shape

    cls_onehot = torch.zeros((B, H, W, C), dtype=torch.float32, device=device)
    if pos_mask.any():
        cls_ids = cls_tgt[pos_mask]  
        cls_onehot[pos_mask] = F.one_hot(cls_ids, num_classes=C).float()

    cls_logits_flat   = cls_pred.permute(0, 2, 3, 1).reshape(-1, C)
    cls_targets_flat  = cls_onehot.reshape(-1, C)
    cls_loss = sigmoid_focal_loss(cls_logits_flat, cls_targets_flat, alpha=0.25, gamma=2.0, reduction="sum")

    if pos_mask.any():
        box_pred_pos = box_pred.permute(0, 2, 3, 1)[pos_mask] 
        reg_tgt_pos  = reg_tgt[pos_mask]                        

        loc_map = locations.view(H, W, 2).unsqueeze(0).expand(B, -1, -1, -1)
        loc_pos = loc_map[pos_mask]

        pred_xyxy = ltrb_to_xyxy(loc_pos, box_pred_pos)
        tgt_xyxy  = ltrb_to_xyxy(loc_pos, reg_tgt_pos)

        reg_loss = giou_loss(pred_xyxy, tgt_xyxy, reduction="sum")

        ctr_pred_pos = ctr_pred.squeeze(1)[pos_mask]  
        ctr_tgt_pos  = ctr_tgt[pos_mask]              
        ctr_loss = F.binary_cross_entropy_with_logits(ctr_pred_pos, ctr_tgt_pos, reduction="sum")
    else:
        reg_loss = torch.tensor(0.0, device=device)
        ctr_loss = torch.tensor(0.0, device=device)

    return cls_loss, reg_loss, ctr_loss

# Assemble the full detector + computing loss

In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class Detector(nn.Module):
    def __init__(self, num_classes: int, base_ch=32, fpn_ch=128):
        super().__init__()
        self.num_classes = int(num_classes)

        self.backbone = TinyBackbone(base_ch=base_ch)
        self.fpn = FPN(in_channels=self.backbone.out_channels, fpn_ch=fpn_ch)
        self.head = DetectionHead(in_ch=fpn_ch, num_classes=self.num_classes)

        self.strides = list(self.backbone.out_strides)

    def forward(self, images):
        c3, c4, c5 = self.backbone(images)
        p3, p4, p5 = self.fpn(c3, c4, c5)
        cls_out, box_out, ctr_out = self.head([p3, p4, p5])
        return (p3, p4, p5), cls_out, box_out, ctr_out


def compute_total_loss(model, images, batch_targets, alpha=1.0, beta=1.0, center_sampling_radius=1.5):

    (p3, p4, p5), cls_out, box_out, ctr_out = model(images)
    features = [p3, p4, p5]

    reg_ranges = default_reg_ranges_from_strides(model.strides, factor=8.0)

    cls_tgts, reg_tgts, ctr_tgts, pos_masks = build_targets_batch(
        features=features,
        targets=batch_targets,
        strides=model.strides,
        num_classes=model.num_classes,
        reg_ranges=reg_ranges,
        center_sampling_radius=center_sampling_radius,
    )

    total_cls = 0.0
    total_reg = 0.0
    total_ctr = 0.0
    total_pos = 0

    for i, (feat, stride) in enumerate(zip(features, model.strides)):
        loc = compute_locations(feat, stride)  

        cls_loss, reg_loss, ctr_loss = compute_losses_per_level(
            cls_pred=cls_out[i],
            box_pred=box_out[i],
            ctr_pred=ctr_out[i],
            cls_tgt=cls_tgts[i],
            reg_tgt=reg_tgts[i],
            ctr_tgt=ctr_tgts[i],
            pos_mask=pos_masks[i],
            locations=loc,
        )

        total_cls += cls_loss
        total_reg += reg_loss
        total_ctr += ctr_loss
        total_pos += int(pos_masks[i].sum().item())

    denom = max(1, total_pos)
    total_cls = total_cls / denom
    total_reg = total_reg / denom
    total_ctr = total_ctr / denom

    total = total_cls + alpha * total_reg + beta * total_ctr

    logs = {
        "loss_total": float(total.item()),
        "loss_cls": float(total_cls.item()),
        "loss_reg": float(total_reg.item()),
        "loss_ctr": float(total_ctr.item()),
        "num_pos": int(total_pos),
    }
    return total, logs



In [ ]:
import torch
from tqdm import tqdm

def run_one_epoch(model, loader, optimizer=None, device="cuda", train=True):
    if train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    total_cls = 0.0
    total_reg = 0.0
    total_ctr = 0.0
    total_pos = 0
    n_batches = 0

    pbar = tqdm(loader, desc="train" if train else "val", leave=False)
    for images, batch_targets in pbar:
        images = images.to(device)

        if train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(train):
            loss, logs = compute_total_loss(model, images, batch_targets, alpha=1.0, beta=1.0)

            if train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                optimizer.step()

        total_loss += logs["loss_total"]
        total_cls  += logs["loss_cls"]
        total_reg  += logs["loss_reg"]
        total_ctr  += logs["loss_ctr"]
        total_pos  += logs["num_pos"]
        n_batches  += 1

        pbar.set_postfix({
            "loss": f"{logs['loss_total']:.4f}",
            "pos": logs["num_pos"]
        })

    return {
        "loss_total": total_loss / max(1, n_batches),
        "loss_cls":   total_cls  / max(1, n_batches),
        "loss_reg":   total_reg  / max(1, n_batches),
        "loss_ctr":   total_ctr  / max(1, n_batches),
        "num_pos":    total_pos,
        "batches":    n_batches,
    }


In [ ]:
import os
class EarlyStopping:
    def __init__(self, patience=8, min_delta=1e-4):
        self.patience = int(patience)
        self.min_delta = float(min_delta)
        self.best = float("inf")
        self.bad = 0

    def step(self, metric):
        improved = metric < (self.best - self.min_delta)
        if improved:
            self.best = metric
            self.bad = 0
        else:
            self.bad += 1
        return improved, (self.bad >= self.patience)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = Detector(num_classes=NUM_CLASSES, base_ch=32, fpn_ch=128).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2,
    threshold=1e-4,
    min_lr=1e-7,
)

early = EarlyStopping(patience=8, min_delta=1e-4)

EPOCHS = 100 

save_dir = "/content/drive/MyDrive/model"
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, "scratch_model_early_stopping.pt")

best_val = float("inf")

for epoch in range(1, EPOCHS + 1):
    train_logs = run_one_epoch(model, train_loader, optimizer=optimizer, device=device, train=True)
    val_logs   = run_one_epoch(model, val_loader,   optimizer=None,     device=device, train=False)

    val_loss = val_logs["loss_total"]

    scheduler.step(val_loss) 
    lr_now = optimizer.param_groups[0]["lr"]

    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), save_path)
        print(f" Saved best model to {save_path} (val_loss={best_val:.4f})")

    _, should_stop = early.step(val_loss)

    print(
        f"Epoch {epoch:03d}/{EPOCHS} | "
        f"lr={lr_now:.2e} | "
        f"train={train_logs['loss_total']:.4f} | "
        f"val={val_loss:.4f} | "
        f"val_pos={val_logs['num_pos']}"
        + (" EARLY STOP" if should_stop else "")
    )

    if should_stop:
        break

# Inference helpers: decode boxes + NMS

In [19]:
def box_iou_xyxy(boxes1, boxes2):
    if boxes1.numel() == 0 or boxes2.numel() == 0:
        return torch.zeros((boxes1.shape[0], boxes2.shape[0]), device=boxes1.device)

    x11, y11, x12, y12 = boxes1[:,0], boxes1[:,1], boxes1[:,2], boxes1[:,3]
    x21, y21, x22, y22 = boxes2[:,0], boxes2[:,1], boxes2[:,2], boxes2[:,3]

    inter_x1 = torch.max(x11[:, None], x21[None, :])
    inter_y1 = torch.max(y11[:, None], y21[None, :])
    inter_x2 = torch.min(x12[:, None], x22[None, :])
    inter_y2 = torch.min(y12[:, None], y22[None, :])

    inter_w = (inter_x2 - inter_x1).clamp(min=0)
    inter_h = (inter_y2 - inter_y1).clamp(min=0)
    inter = inter_w * inter_h

    area1 = (x12 - x11).clamp(min=0) * (y12 - y11).clamp(min=0)
    area2 = (x22 - x21).clamp(min=0) * (y22 - y21).clamp(min=0)

    union = area1[:, None] + area2[None, :] - inter + 1e-6
    return inter / union


def nms_xyxy(boxes, scores, iou_thresh=0.5):
    if boxes.numel() == 0:
        return torch.empty((0,), dtype=torch.long, device=boxes.device)

    order = scores.argsort(descending=True)
    keep = []

    while order.numel() > 0:
        i = order[0].item()
        keep.append(i)

        if order.numel() == 1:
            break

        rest = order[1:]
        ious = box_iou_xyxy(boxes[i].unsqueeze(0), boxes[rest]).squeeze(0)
        order = rest[ious <= iou_thresh]

    return torch.tensor(keep, dtype=torch.long, device=boxes.device)


def decode_level_boxes(locations, ltrb):
    x = locations[:, 0]
    y = locations[:, 1]
    l = ltrb[:, 0]
    t = ltrb[:, 1]
    r = ltrb[:, 2]
    b = ltrb[:, 3]
    return torch.stack([x - l, y - t, x + r, y + b], dim=1)


@torch.no_grad()
def predict_single_image(
    model,
    image_tensor,        
    score_thresh=0.30,
    iou_thresh=0.50,
    max_dets=200,
):
    device = next(model.parameters()).device
    model.eval()
    image_tensor = image_tensor.to(device)

    (p3, p4, p5), cls_out, box_out, ctr_out = model(image_tensor)

    feats = [p3, p4, p5]
    strides = list(model.strides)
    num_classes = int(model.num_classes)

    assert len(feats) == len(strides) == len(cls_out) == len(box_out) == len(ctr_out), \
        "Mismatch between number of feature levels and outputs/strides."

    all_boxes, all_scores, all_labels = [], [], []

    for lvl in range(len(feats)):  
        cls_logits = cls_out[lvl][0]     
        ltrb = box_out[lvl][0]           
        ctr_logits = ctr_out[lvl][0, 0]  

        H, W = cls_logits.shape[1], cls_logits.shape[2]
        L = H * W

        cls_logits = cls_logits.permute(1, 2, 0).reshape(L, num_classes)  
        ltrb = ltrb.permute(1, 2, 0).reshape(L, 4)                        
        ctr = torch.sigmoid(ctr_logits).reshape(L)                        

        cls_prob = torch.sigmoid(cls_logits)                              
        cls_score, cls_label = cls_prob.max(dim=1)                        
        score = cls_score * ctr                                           

        keep = score >= score_thresh
        if keep.sum() == 0:
            continue

        cls_label = cls_label[keep]
        score = score[keep]
        ltrb = ltrb[keep]

        locations = compute_locations(feats[lvl], strides[lvl])  
        locations = locations[keep]                              

        boxes = decode_level_boxes(locations, ltrb)              

        all_boxes.append(boxes)
        all_scores.append(score)
        all_labels.append(cls_label)

    if not all_boxes:
        return (
            torch.zeros((0, 4), device=device),
            torch.zeros((0,), device=device),
            torch.zeros((0,), dtype=torch.long, device=device),
        )

    boxes = torch.cat(all_boxes, dim=0)
    scores = torch.cat(all_scores, dim=0)
    labels = torch.cat(all_labels, dim=0)

    _, _, Himg, Wimg = image_tensor.shape
    boxes[:, 0].clamp_(0, Wimg - 1)
    boxes[:, 2].clamp_(0, Wimg - 1)
    boxes[:, 1].clamp_(0, Himg - 1)
    boxes[:, 3].clamp_(0, Himg - 1)

    final_keep = []
    for c in range(num_classes):
        idx = torch.where(labels == c)[0]
        if idx.numel() == 0:
            continue
        keep_idx = nms_xyxy(boxes[idx], scores[idx], iou_thresh=iou_thresh)
        final_keep.append(idx[keep_idx])

    if final_keep:
        final_keep = torch.cat(final_keep)
        top = scores[final_keep].argsort(descending=True)[:max_dets]
        final_keep = final_keep[top]
    else:
        final_keep = torch.empty((0,), dtype=torch.long, device=device)

    return boxes[final_keep], scores[final_keep], labels[final_keep]


# Testing

In [24]:
import numpy as np

def iou_one_to_many(box, boxes):
    if boxes.numel() == 0:
        return torch.zeros((0,), device=boxes.device)

    x1 = torch.maximum(box[0], boxes[:, 0])
    y1 = torch.maximum(box[1], boxes[:, 1])
    x2 = torch.minimum(box[2], boxes[:, 2])
    y2 = torch.minimum(box[3], boxes[:, 3])

    inter = (x2 - x1).clamp(min=0) * (y2 - y1).clamp(min=0)

    area_box = (box[2] - box[0]).clamp(min=0) * (box[3] - box[1]).clamp(min=0)
    area_boxes = (boxes[:, 2] - boxes[:, 0]).clamp(min=0) * (boxes[:, 3] - boxes[:, 1]).clamp(min=0)

    union = area_box + area_boxes - inter + 1e-6
    return inter / union


def compute_ap_from_pr(prec, rec):
    ap = 0.0
    for t in np.linspace(0, 1, 11):
        p = prec[rec >= t].max() if np.any(rec >= t) else 0.0
        ap += p / 11.0
    return ap


@torch.no_grad()
def evaluate_map50(model, test_loader, num_classes=None, score_thresh=0.05, iou_thresh=0.50, nms_iou=0.50):
    device = next(model.parameters()).device
    model.eval()

    if num_classes is None:
        num_classes = int(model.num_classes)
    else:
        num_classes = int(num_classes)

    pred_records = {c: [] for c in range(num_classes)}
    gt_count = {c: 0 for c in range(num_classes)}

    for images, targets in test_loader:
        B = images.shape[0]
        for b in range(B):
            img = images[b:b+1].to(device)
            gt_boxes = targets[b]["boxes"].to(device)
            gt_labels = targets[b]["labels"].to(device)

            for c in range(num_classes):
                gt_count[c] += int((gt_labels == c).sum().item())

            boxes_p, scores_p, labels_p = predict_single_image(
                model, img, score_thresh=score_thresh, iou_thresh=nms_iou
            )

            for c in range(num_classes):
                gt_idx = torch.where(gt_labels == c)[0]
                pred_idx = torch.where(labels_p == c)[0]

                gt_c = gt_boxes[gt_idx] if gt_idx.numel() else gt_boxes.new_zeros((0, 4))
                pred_c = boxes_p[pred_idx] if pred_idx.numel() else boxes_p.new_zeros((0, 4))
                score_c = scores_p[pred_idx] if pred_idx.numel() else scores_p.new_zeros((0,))

                matched = torch.zeros((gt_c.shape[0],), dtype=torch.bool, device=device)

                if score_c.numel():
                    order = torch.argsort(score_c, descending=True)
                    pred_c = pred_c[order]
                    score_c = score_c[order]

                for j in range(pred_c.shape[0]):
                    if gt_c.shape[0] == 0:
                        pred_records[c].append((float(score_c[j].item()), 0))  # FP
                        continue

                    ious = iou_one_to_many(pred_c[j], gt_c)
                    best_iou, best_k = ious.max(dim=0)

                    if best_iou >= iou_thresh and not matched[best_k]:
                        matched[best_k] = True
                        pred_records[c].append((float(score_c[j].item()), 1))  # TP
                    else:
                        pred_records[c].append((float(score_c[j].item()), 0))  # FP

    ap_per_class = {}
    pr_per_class = {}

    for c in range(num_classes):
        recs = pred_records[c]

        if gt_count[c] == 0:
            ap_per_class[c] = float("nan")
            pr_per_class[c] = (0.0, 0.0)
            continue

        if len(recs) == 0:
            ap_per_class[c] = 0.0
            pr_per_class[c] = (0.0, 0.0)
            continue

        recs.sort(key=lambda x: x[0], reverse=True)
        tps = np.array([r[1] for r in recs], dtype=np.float32)
        fps = 1.0 - tps

        tp_cum = np.cumsum(tps)
        fp_cum = np.cumsum(fps)

        prec = tp_cum / np.maximum(tp_cum + fp_cum, 1e-6)
        rec  = tp_cum / max(gt_count[c], 1e-6)

        ap = compute_ap_from_pr(prec, rec)
        ap_per_class[c] = float(ap)
        pr_per_class[c] = (float(prec[-1]), float(rec[-1]))

    aps = [v for v in ap_per_class.values() if not np.isnan(v)]
    mAP50 = float(np.mean(aps)) if aps else 0.0

    return mAP50, ap_per_class, pr_per_class, gt_count


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Detector(num_classes=NUM_CLASSES, base_ch=32, fpn_ch=128).to(device)
model.load_state_dict(torch.load("/content/drive/MyDrive/model/scratch_model_early_stopping.pt", map_location=device))
model.eval()

mAP50, ap_per_class, pr_per_class, gt_count = evaluate_map50(
    model, test_loader,
    num_classes=None,
    score_thresh=0.2,
    iou_thresh=0.5,
    nms_iou=0.50
)

print("GT count per class:", gt_count)
print("AP@0.5 per class:", ap_per_class)
print("Precision/Recall per class (end):", pr_per_class)
print("mAP@0.5:", mAP50)


In [ ]:
id_to_name

In [26]:
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont

def tensor_to_pil(img_tensor):
    img = (img_tensor.clamp(0, 1) * 255).byte().permute(1, 2, 0).cpu().numpy()
    return Image.fromarray(img)

def draw_xyxy(img_pil, boxes, labels, scores=None, id_to_name=None, color=(255, 0, 0), width=4, draw_text=True):
    img = img_pil.copy()
    draw = ImageDraw.Draw(img)

    try:
        font = ImageFont.truetype("DejaVuSans.ttf", 18)
    except:
        font = ImageFont.load_default()

    for i in range(boxes.shape[0]):
        x1, y1, x2, y2 = boxes[i].tolist()
        draw.rectangle([x1, y1, x2, y2], outline=color, width=width)

        if draw_text:
            cls = int(labels[i].item())
            name = str(cls) if id_to_name is None else id_to_name.get(cls, str(cls))
            txt = f"{name}" if scores is None else f"{name} {float(scores[i]):.2f}"

            tb = draw.textbbox((0, 0), txt, font=font)
            tw, th = tb[2] - tb[0], tb[3] - tb[1]
            tx, ty = x1, max(0, y1 - th - 8)
            draw.rectangle([tx, ty, tx + tw + 10, ty + th + 6], fill=color)
            draw.text((tx + 5, ty + 3), txt, fill=(255, 255, 255), font=font)

    return img

@torch.no_grad()
def visualize_all_test(
    model,
    loader,
    score_thresh=0.30,
    iou_thresh=0.50,
    max_images=20,
):
    device = next(model.parameters()).device
    model.eval()

    shown = 0
    for images, targets in loader:
        B = images.shape[0]
        for b in range(B):
            if max_images is not None and shown >= max_images:
                return

            img_tensor = images[b:b+1].to(device) 
            base_img = tensor_to_pil(images[b])    

            gt_boxes = targets[b]["boxes"]
            gt_labels = targets[b]["labels"]

            gt_names = []
            for c in gt_labels.tolist():
                gt_names.append(id_to_name.get(int(c), str(int(c))))
            gt_names = sorted(set(gt_names))
            gt_title = ", ".join(gt_names) if gt_names else "no_gt"

            boxes_p, scores_p, labels_p = predict_single_image(
                model, img_tensor, score_thresh=score_thresh, iou_thresh=iou_thresh
            )

            vis = draw_xyxy(
                base_img,
                gt_boxes,
                gt_labels,
                scores=None,
                id_to_name=id_to_name,
                color=(0, 180, 0),
                width=3,
                draw_text=False,  
            )

            if boxes_p.numel() > 0:
                vis = draw_xyxy(
                    vis,
                    boxes_p.cpu(),
                    labels_p.cpu(),
                    scores=scores_p.cpu(),
                    id_to_name=id_to_name,
                    color=(255, 0, 0),
                    width=4,
                    draw_text=True, 
                )

            plt.figure(figsize=(8, 8))
            plt.imshow(vis)
            plt.axis("off")
            plt.title(f"img {shown} | {gt_title}")  
            plt.show()

            shown += 1

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Detector(num_classes=NUM_CLASSES, base_ch=32, fpn_ch=128).to(device)
model.load_state_dict(torch.load("/content/drive/MyDrive/model/scratch_model_early_stopping.pt", map_location=device))

visualize_all_test(model, test_loader, score_thresh=0.30, iou_thresh=0.50, max_images=900)


# Check Point

In [ ]:
import torch
import os

def save_checkpoint(
    path,
    epoch,
    model,
    optimizer,
    scheduler,
    best_val,
    early_stopper,
):
    checkpoint = {
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict() if scheduler is not None else None,
        "best_val": best_val,
        "early_state": {
            "best": early_stopper.best,
            "bad": early_stopper.bad,
        },
    }
    torch.save(checkpoint, path)

In [ ]:
CHECKPOINT_DIR = Path("/content/drive/MyDrive/model/checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

checkpoint = os.path.join(CHECKPOINT_DIR, "last.ckpt")


In [ ]:
save_checkpoint(checkpoint, epoch, model, optimizer, scheduler, best_val, early)